# Actividad — Clasificación de correos spam con regresión logística y kNN
**Aprendizaje Supervisado · Sesión 8 · Regresión logística y $k$ vecinos más cercanos**

**Qué haremos.** En la presentación vimos cómo la regresión logística convierte una combinación lineal
de variables en una probabilidad con la función **sigmoide**, y cómo **kNN** clasifica un punto nuevo
por mayoría de voto entre sus $k$ vecinos más cercanos. Hoy comparas ambos enfoques en un problema
real: decidir si un correo es **spam** o no, a partir de ocho variables numéricas derivadas de su
contenido (número de palabras, proporción de mayúsculas, signos de exclamación, enlaces, etc. — en el
espíritu del clásico conjunto *Spambase*).

Trabaja **de forma individual**. Las celdas con `...` están incompletas: reemplaza los puntos
suspensivos con el código que falta (el comentario `✏️` te da una pista de qué se espera).


## Parte 0 — Los datos

Cada fila es un correo con 8 variables de entrada (`num_palabras`, `prop_mayusculas`,
`num_signos_exclamacion`, `num_palabras_dinero`, `num_enlaces`, `num_mayus_consecutivas`,
`remitente_conocido`, `num_adjuntos`) y la etiqueta `spam` (1 = spam, 0 = no spam). El riesgo real
depende sobre todo de la proporción de mayúsculas, las palabras asociadas a dinero/ofertas, los signos
de exclamación y los enlaces; `num_adjuntos` casi no influye (es la variable "de control").


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})


def genera_correos(n, semilla):
    # 8 variables numéricas derivadas del contenido de un correo (estilo Spambase) y la
    # etiqueta spam (1/0); num_adjuntos es una variable "de control" que casi no influye.
    rng = np.random.default_rng(semilla)
    num_palabras = np.clip(rng.gamma(6.0, 20.0, n), 15, 400).round(0)
    prop_mayusculas = np.clip(rng.beta(1.6, 9.0, n), 0, 1)
    signos_exclamacion = rng.poisson(0.6, n).clip(0, 20).astype(float)
    palabras_dinero = rng.poisson(0.5, n).clip(0, 15).astype(float)
    num_enlaces = rng.poisson(0.8, n).clip(0, 12).astype(float)
    mayus_consecutivas = rng.poisson(0.4, n).clip(0, 10).astype(float)
    remitente_conocido = rng.binomial(1, 0.55, n).astype(float)
    num_adjuntos = rng.poisson(0.3, n).clip(0, 6).astype(float)
    z = (-4.6 + 1.75 * palabras_dinero + 1.05 * signos_exclamacion + 0.85 * num_enlaces
         + 6.5 * prop_mayusculas + 0.65 * mayus_consecutivas - 2.2 * remitente_conocido
         + 0.012 * num_palabras + 0.05 * num_adjuntos)
    p = 1 / (1 + np.exp(-z))
    spam = (rng.uniform(0, 1, n) < p).astype(int)
    return pd.DataFrame({"num_palabras": num_palabras, "prop_mayusculas": prop_mayusculas.round(3),
                          "num_signos_exclamacion": signos_exclamacion, "num_palabras_dinero": palabras_dinero,
                          "num_enlaces": num_enlaces, "num_mayus_consecutivas": mayus_consecutivas,
                          "remitente_conocido": remitente_conocido.astype(int), "num_adjuntos": num_adjuntos,
                          "spam": spam})


correos = genera_correos(3000, semilla=2026)
print(correos.shape)
correos.head()


In [ ]:
conteo = correos["spam"].value_counts()
prop = correos["spam"].value_counts(normalize=True)
print(conteo)
print((prop * 100).round(1), "%")

fig, ax = plt.subplots(figsize=(4, 3.2))
ax.bar(["no spam (0)", "spam (1)"], conteo.values, color=["#1C7293", "#E07A1F"])
ax.set_ylabel("número de correos"); ax.set_title("Balance de clases")
for i, v in enumerate(conteo.values):
    ax.text(i, v + 20, str(v), ha="center")
plt.show()


✏️ **Pregunta 0.1.** (a) ¿Cuántos correos hay de cada clase y qué proporción representa el spam?
(b) ¿Dirías que el conjunto está desbalanceado? ¿Qué tan grave es ese desbalance comparado, por
ejemplo, con un conjunto donde el 2% de los casos es la clase positiva?

**Respuesta:**


## Parte 1 — Partición y regresión logística  *(presentación A1–A9)*

Dividimos en entrenamiento/prueba de forma **estratificada** (para conservar la proporción de spam en
ambos conjuntos) y ajustamos un `Pipeline` de `StandardScaler` + `LogisticRegression`, eligiendo $C$
con `GridSearchCV` sobre validación cruzada de 5 pliegues.


In [ ]:
X = correos.drop(columns="spam")
y = correos["spam"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print("Proporción de spam -> entrenamiento:", round(ytr.mean(), 3), "  prueba:", round(yte.mean(), 3))

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
malla_C = {"logisticregression__C": np.logspace(-3, 3, 13)}
pipe_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
busq_lr = ...     # ✏️ GridSearchCV(pipe_lr, malla_C, cv=kf, scoring="accuracy").fit(Xtr, ytr)

modelo_lr = busq_lr.best_estimator_
print("Mejor C:", busq_lr.best_params_["logisticregression__C"],
      "   exactitud de validación:", round(busq_lr.best_score_, 3))
print("Exactitud en prueba:", round(modelo_lr.score(Xte, yte), 3))


In [ ]:
coefs = pd.Series(modelo_lr[-1].coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print(coefs.round(2).to_string())


✏️ **Pregunta 1.1.** (a) ¿Qué $C$ eligió `GridSearchCV` y qué exactitud de prueba obtuvo el modelo?
(b) Mirando los coeficientes (ya estandarizados, así que son comparables entre sí): ¿qué dos o tres
variables pesan más y con qué signo? (c) `num_adjuntos` se diseñó para que casi no influyera en la
etiqueta real: ¿se refleja eso en su coeficiente?

**Respuesta:**


## Parte 2 — $k$ vecinos más cercanos  *(presentación C1–C7)*

Igual que con la regresión logística, el escalado es **obligatorio** en kNN porque el algoritmo mide
distancias. Elegimos $k$ con una curva de validación cruzada sobre valores impares de 1 a 39.


In [ ]:
valores_k = list(range(1, 40, 2))
exact_cv = []
for k in valores_k:
    pipe_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    exact_cv.append(...     # ✏️ cross_val_score(pipe_knn, Xtr, ytr, cv=kf, scoring="accuracy").mean())

mejor_k = valores_k[int(np.argmax(exact_cv))]
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(valores_k, exact_cv, "o-", color="#065A82")
ax.axvline(mejor_k, color="#E07A1F", ls="--", lw=1)
ax.set_xlabel("k"); ax.set_ylabel("exactitud (validación cruzada)")
ax.set_title(f"Mejor k = {mejor_k}  (exactitud = {max(exact_cv):.3f})")
plt.show()

modelo_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=mejor_k)).fit(Xtr, ytr)
print("Exactitud en prueba (kNN):", round(modelo_knn.score(Xte, yte), 3))


✏️ **Pregunta 2.1.** (a) ¿Qué $k$ resultó mejor y qué exactitud de validación tiene? (b) ¿Es mayor o
menor que la exactitud de prueba de la regresión logística de la Parte 1? (c) ¿Por qué habría sido un
error usar `KNeighborsClassifier` directamente sobre `num_palabras` y `prop_mayusculas` **sin**
escalar, si estas dos variables tienen escalas tan distintas (decenas/cientos frente a 0–1)?

**Respuesta:**


## Parte 3 — Comparación final: matriz de confusión y métricas  *(presentación D1–D5)*

Comparamos los dos modelos ya ajustados con `classification_report` y su matriz de confusión, **una
sola vez**, sobre el conjunto de prueba.


In [ ]:
for nombre, modelo in [("Regresión logística", modelo_lr), ("kNN", modelo_knn)]:
    print(f"\n=== {nombre} ===")
    print(classification_report(yte, modelo.predict(Xte), target_names=["no spam", "spam"]))

fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, (nombre, modelo) in zip(axs, [("Regresión logística", modelo_lr), ("kNN", modelo_knn)]):
    ConfusionMatrixDisplay.from_estimator(modelo, Xte, yte, display_labels=["no spam", "spam"],
                                           cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(nombre)
plt.tight_layout(); plt.show()


✏️ **Pregunta 3.1.** Lee ambas matrices de confusión (filas = verdad, columnas = predicción). (a) ¿Qué
modelo comete más **falsos positivos** (correos buenos marcados como spam)? (b) ¿Qué modelo comete más
**falsos negativos** (spam que se les escapa)? (c) En términos de F1 para la clase "spam", ¿cuál
modelo es mejor en conjunto?

**Respuesta:**


## Parte 4 — Recomendación final

Imagina que tienes que entregar uno de los dos modelos como filtro de spam para un servidor de correo
real.


✏️ **Pregunta 4.1.** En 5 a 7 líneas, recomienda un modelo para un filtro de spam real: (1) cuál
recomendarías entre regresión logística y kNN, con sus cifras de precisión y recall para la clase
spam; (2) en un filtro de correo, ¿qué error suele ser más costoso para el usuario: un falso positivo
(perder un correo bueno) o un falso negativo (recibir spam)?; (3) cómo influye esa prioridad en tu
elección entre los dos modelos de este cuaderno, dadas sus cifras.

**Respuesta:**
